# 26 — Cross-Wind Plume Tomography

Lightweight sector-based screening analogue of plume tomography.

In [ ]:

BACK_TRAJ_CSV = "outputs/back_trajectory/NHV_back_trajectory.csv"
OUTPUT_DIR = "outputs/crosswind_tomography"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

traj, meta = safe_read_csv(BACK_TRAJ_CSV)
if not traj.empty:
    traj["lat"] = pd.to_numeric(traj["lat"], errors="coerce")
    traj["lon"] = pd.to_numeric(traj["lon"], errors="coerce")
REF_LAT = 50.80208; REF_LON = 0.04961
def bearing_sector(lat, lon):
    dy = lat - REF_LAT; dx = lon - REF_LON
    ang = (math.degrees(math.atan2(dx, dy)) + 360) % 360
    bins = [(337.5,360,"N"),(0,22.5,"N"),(22.5,67.5,"NE"),(67.5,112.5,"E"),(112.5,157.5,"SE"),(157.5,202.5,"S"),(202.5,247.5,"SW"),(247.5,292.5,"W"),(292.5,337.5,"NW")]
    for a,b,name in bins:
        if a <= ang < b: return name
    return "UNK"
traj["sector"] = [bearing_sector(lat, lon) for lat, lon in zip(traj.get("lat", []), traj.get("lon", []))] if not traj.empty else []
sector = traj.groupby("sector", dropna=False).agg(steps=("step","count"), mean_lat=("lat","mean"), mean_lon=("lon","mean")).reset_index().sort_values("steps", ascending=False) if not traj.empty else pd.DataFrame()
traj.to_csv(Path(OUTPUT_DIR) / "crosswind_tomography_points.csv", index=False)
sector.to_csv(Path(OUTPUT_DIR) / "crosswind_tomography_sector_summary.csv", index=False)
display(sector.head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
if not sector.empty: ax.bar(sector["sector"], sector["steps"])
ax.set_title("Cross-wind tomography sector occupancy"); ax.set_xlabel("Sector"); ax.set_ylabel("Trajectory steps")
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "crosswind_tomography_plot.png"; fig.savefig(plot_path, dpi=150); plt.show()
(Path(OUTPUT_DIR) / "crosswind_tomography_manifest.json").write_text(json.dumps({"input": meta, "rows_traj": int(len(traj)), "rows_sector": int(len(sector))}, indent=2), encoding="utf-8")
print("Saved:", plot_path)
